## Models

This notebook trains and compares a Logistic Regression baseline and a Random Forest model, optimizing for recall, F1, and AUC-PR while monitoring the precision tradeoff. These metrics were chosen specifically because of the severe class imbalance identified in the data overview; accuracy alone would be misleading at a 0.17% fraud rate. Both models use balanced class weights to compensate for the imbalance during training. All data has been preprocessed and is ready to load directly from the database.

In [29]:
import pandas as pd
import numpy as np
import sqlite3
from sklearn.linear_model import LogisticRegression

conn = sqlite3.connect("../fraud.db")
X_train = pd.read_sql(""" SELECT * FROM X_train""", conn)
X_test = pd.read_sql(""" SELECT * FROM X_test""", conn)
y_train = pd.read_sql(""" SELECT * FROM y_train""", conn)['Class']
y_test = pd.read_sql(""" SELECT * FROM y_test""", conn)['Class']
conn.close()

In [30]:
model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000).fit(X_train, y_train)

In [31]:
predictions = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]

print(predictions[:5])
print(y_test[:5])

print(probs)

[0 0 0 0 1]
0    0
1    0
2    0
3    0
4    0
Name: Class, dtype: int64
[0.00584217 0.06918075 0.00013404 ... 0.00027176 0.00392062 0.08460862]


In [32]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

report = classification_report(y_test, predictions, output_dict=True)
conf_mat = confusion_matrix(y_test, predictions)
roc_score = roc_auc_score(y_test, probs)
pr_score = average_precision_score(y_test, probs)

print(f"CLASSIFICATION REPORT: {report['1']}")
print(f"CONFUSION MATRIX: {conf_mat}")
print(f"AUC-ROC:  {roc_score:.4f}")
print(f"AUC-PR:   {pr_score:.4f}")

CLASSIFICATION REPORT: {'precision': 0.06097560975609756, 'recall': 0.9183673469387755, 'f1-score': 0.11435832274459974, 'support': 98.0}
CONFUSION MATRIX: [[55478  1386]
 [    8    90]]
AUC-ROC:  0.9722
AUC-PR:   0.7159


In [33]:
results = pd.DataFrame([{
    'model': "Logistic Regression",
    'recall': report['1']['recall'],
    'precision': report['1']['precision'],
    'f1': report['1']['f1-score'],
    'auc_roc' : roc_score,
    'auc_pr': pr_score
}])

conn = sqlite3.connect("../fraud.db")
results.to_sql('model_results', conn, if_exists='replace', index=False)
conn.close()

### Review of results from Logistic Regression Model

The Logistic Regression model proved to be a strong baseline for our fraud detection. It achieved a 0.918 Recall score and a 0.972 AUC-ROC score. However, where it failed was in the precision, generating a significant amount of false positives. The High AUC-PR 0.716, shows that a meaningful signal exists in the features. Next, we'll implmement a non-linear model, in hopes of improving the precision-recall balance

## Random Forest Model
Logistic Regression maximized recall at the cost of precision, generating significant false positive noise. Random Forest dramatically improved precision and F1, but at the cost of missing more fraud cases at the default threshold. This tradeoff suggests that threshold tuning on the Random Forest offers a path to recovering recall without sacrificing the precision gains.

The following section tunes the decision threshold to find the optimal balance between catching fraud and minimizing false alarms.

In [34]:
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1).fit(X_train, y_train)

In [37]:
forest_predictions = random_forest.predict(X_test)
forest_probs = random_forest.predict_proba(X_test)[:,1]

print(forest_predictions[:5])
print(y_test[:5])

print(forest_probs[:5])

[0 0 0 0 0]
0    0
1    0
2    0
3    0
4    0
Name: Class, dtype: int64
[0. 0. 0. 0. 0.]


In [38]:
forest_report = classification_report(y_test, forest_predictions, output_dict=True)
forest_conf_mat = confusion_matrix(y_test, forest_predictions)
forest_roc_score = roc_auc_score(y_test, forest_probs)
forest_pr_score = average_precision_score(y_test, forest_probs)

print(f"CLASSIFICATION REPORT: {forest_report['1']}")
print(f"CONFUSION MATRIX: {forest_conf_mat}")
print(f"AUC-ROC:  {forest_roc_score:.4f}")
print(f"AUC-PR:   {forest_pr_score:.4f}")

CLASSIFICATION REPORT: {'precision': 0.9605263157894737, 'recall': 0.7448979591836735, 'f1-score': 0.8390804597701149, 'support': 98.0}
CONFUSION MATRIX: [[56861     3]
 [   25    73]]
AUC-ROC:  0.9529
AUC-PR:   0.8542


## Threshold Tuning

With the decision threshold lowered to 0.3, the Random Forest recovered 10 additional fraud cases at the cost of only 4 additional false positives. This threshold was selected based on precision-recall tradeoff analysis; below 0.3, false positives increased sharply with minimal recall gains.

In fraud detection, a missed fraud case carries a higher cost than a false alarm; the financial and reputational damage of undetected fraud outweighs the friction of occasionally flagging a legitimate transaction. This makes the tradeoff acceptable and 0.3 the preferred operating threshold.

In [47]:
threshold = 0.3
predictions_adjusted = (forest_probs >= threshold).astype(int)

forest_report_threshold = classification_report(y_test, predictions_adjusted, output_dict=True)
forest_conf_mat_threshold = confusion_matrix(y_test, predictions_adjusted)


print(f"CLASSIFICATION REPORT: {forest_report_threshold['1']}")
print(f"CONFUSION MATRIX: {forest_conf_mat_threshold}")


CLASSIFICATION REPORT: {'precision': 0.9222222222222223, 'recall': 0.8469387755102041, 'f1-score': 0.8829787234042553, 'support': 98.0}
CONFUSION MATRIX: [[56857     7]
 [   15    83]]


In [48]:
results = pd.DataFrame([{
    'model': "Random Forest",
    'recall': forest_report_threshold['1']['recall'],
    'precision': forest_report_threshold['1']['precision'],
    'f1': forest_report_threshold['1']['f1-score'],
    'auc_roc' : forest_roc_score,
    'auc_pr': forest_pr_score
}])

conn = sqlite3.connect("../fraud.db")
results.to_sql('model_results', conn, if_exists='append', index=False)
conn.close()

## Conclusion

The Logistic Regression baseline achieved strong recall (0.918) but generated significant false positives, resulting in poor precision (0.061) and a low F1 score (0.114). After threshold tuning, the Random Forest achieved a strong balance between recall (0.847) and precision (0.922), producing a substantially higher F1 score (0.883).

For a fraud detection system, I recommend the tuned Random Forest. It catches the majority of fraudulent transactions while minimizing false alarms for legitimate customers; this reduces friction for cardholders without compromising protection. These findings and model comparisons will be summarized in the Tableau presentation layer